## Atmospheric Radiation Dose Rate

The amount of ionizing radiation energy deposited in an area per unit time. It is the sum of cosmic radiation, terrestrial radiation, and artificial radiation.

Units:

- nanoSieverts per hour (nSv/hr)
- microGrays per hour (µGy/hr)

Depends on:

- Latitude  
- Altitude  
- Geomagnetic conditions  
- Solar activity  

### Cosmic Rays

High energy particles originating from outer space are called cosmic rays.

1. **Galactic CRs**: Originates from the galaxy. They have almost omnipresent and isotropic flux near Earth (the magnitude is invariant with the angle of observation). 
     - Energy Range: Several GeV/nucleon to $10^{20}$ eV per nucleon
     - Consists of about 90% protons, 9% helium, and 1% heavy element nuclei near Earth

2. **Solar Energetic Particles (SEPs)**: High energy particles are accelerated near the sun in solar eruptive events such as flares or coronal mass ejections.
     - Sporadic occurence and very strong variability
     - Energy Range: Below 100 MeV (several GeV in rare very energetic events)

3. **Anomalous CRs**: Originating from interstellar neutral atoms. Very low energy and are not affected by atmosphere.

*References*

[1] I.G. Usoskin, G.A. Kovaltsov and A.L. Mishev. J. Space Weather Space Clim., 14 (2024) 20 DOI: [https://doi.org/10.1051/swsc/2024020](https://doi.org/10.1051/swsc/2024020)

[2] Usoskin, I. G., and G. A.Kovaltsov (2006), Cosmic ray induced ionization in the atmosphere: Full modeling and practical applications, J. Geophys. Res., 111, D21206, [https://doi:10.1029/2006JD007150](https://doi:10.1029/2006JD007150).

[3] Larsen, N., Mishev, A., & Usoskin, I. (2023). *JGR Space Physics*, 128, e2022JA031061. [https://doi.org/10.1029/2022JA031061](https://doi.org/10.1029/2022JA031061)

## Cosmic Ray Atmospheric Cascade: Cosmic Ray Induced Ionization (CRAC:CRII)

### CR interaction with Earth's atmosphere

Atmospheric ionization and Radiation dose are a result of the interaction of GCRs (protons and alpha particles) with the Earth's atmosphere, producing a cascade of secondary particles.

The flux of these particles reaching a location is determined by the cutoff Rigidity or the geomagnetic shielding.

The cutoff rigidity $R_c$ is defined as the minimum rigidity required for a particle to penetrate the geomagnetic field. 
 
$R_c$ is a function of latitude $(\lambda)$, longitude $(\phi)$, altitude $(h)$, time $(t)$, geomagnetic activity index $(K_p)$, and a field model (internal MF from Earth's core - `IGRF` and extrenal MF from magnetospheric currents - `Tsyganenko 89`)

$$
R_c = f(\lambda, \phi, h, t, K_p, \text{field model})
$$

> $K_p$: It is geomagnetic index (0 - 9) that measures the magnetospheric disturbances driven by solar wind and internal magnetic field. High $K_p$ (during a solar storm) results in a compressed magnetosphere, lower effective cutoff, resulting in more particle penetration.

$R_c$ is calculated using the OTSO model [3]

### Cosmic Ray Spectrum (Using Force-Field approximation)

According to [2], the differential energy spectrum of cosmic rays is modeled as: 

$$
J(E,\phi) = J_0 \cdot (E+\phi)^{-\gamma}
$$

where, E: Kinetic Energy (GeV), $\phi$: solar modulation potential (MV), $\gamma \approx 2.7$

### Yield Function from CRAC Model $(Y(E,h))$

The yield function in CRAC model represents ion pairs per primary particle (ionization rate), or the dose contribution per particle.

- Obtained from Monte Carlo Simulations (GEANT4 - toolkit for the simulation of the passage of particles through matter).

### Atmospheric Ionization / Dose Rate

It is the integration of the energy from the cosmic rays and yield function over energy element $dE$.

$$
Q(h) = \int_{E_{min}}^{\infty} J(E)\cdot Y(E,h)dE
$$

$E_{min}$ corresponds to the energy at $R_c$.

**Rigidity-Energy relation for protons**: 

$$
R = \sqrt{E(E+2m_p)} \\
\implies E = \sqrt{R^2+m_p^2}- m_p
$$

where, $m_p = 0.938$ GeV

### What happens during an Excursion?

The rigidity cutoff at a specific location drops significantly ($\alpha \in [0.1, 0.3]$), leading to higher radiation at low latitude and altitudes.

$$
R^{exc}_c \ll R^{present}_c \\
R^{exc}_c = \alpha R^{present}_c; \alpha \in [0.1, 0.3]
$$

In [2]:
import numpy as np

## Rigidity - Energy conversions

m_p = 0.938  # mass of proton in GeV

def rigidity_to_energy(R):
    E = np.sqrt(R**2 + m_p**2) - m_p
    return E

def energy_to_rigidity(E):
    R = np.sqrt(E * (E + 2 * m_p))
    return R

In [3]:
# obtain the CR energy spectrum relation

def cr_energy_spectrum(E, phi = 500):
    J = (E + phi/1000.0)**-(2.7)
    return J

[4] Pre-calculated tables of cosmic ray induced ionization (CRII) in units of [ion pairs /g /sec]: `data/CRII_tables/README.txt` - [https://cosmicrays.oulu.fi/CRII/CRII.html](https://cosmicrays.oulu.fi/CRII/CRII.html)

The tables are given in ASCII files with names `S_XXXXXX.RES`, where XXXXXX stands for the residual atmospheric depth in [0.01 g/cm2] (e.g., S_000010.RES corresponds to 0.1 g/cm2 and S_011000.RES to 110 g/cm2). Each file is organized as follows:

- First line: values of the modulation potential Phi (see comment below) in MV from 0 to 1500 MV.
- Following lines give: the geomagnetic cutoff rigidity Pc in GV (first number) followed by CRII for the corresponding values of Pc (row) and Phi (column).

CRII is given as the ionization rate (number of ion pairs per gram of air per second, in order to obtain the ionization rate per cm3 per second, this value should be multiplied by the air density) caused by galactic cosmic rays at given atmospheric height (atmospheric depth corresponding to the file's name as described above), location (via the geomagnetic cutoff rigidity) and time (via the heliospheric modulation of cosmic rays)

We should determine the scale of height with 0.1 g/s to match the data in the CRII tables.

**Assumption**

1. Consider only heights close to the surface so that we can assume a constant temperature (isotherrmal condition) 

Applying the ideal gas law $(P=\rho RT)$ and hydrostatic equilibrium ($\frac{dP}{dh} = -\rho g$), we can obtain -

$$
P(h) = P_0e^{-h/H} \\
\text{and }H = \frac{RT}{g} = \frac{k_BT}{mg}
$$

Using the below derivation, the value of H at $T = 288K$ is obtained

In [6]:
import numpy as np

kB = 1.380649e-23   # Boltzmann constant (J/K)
T = 288            # Temperature (K)
m = 29 * 1.6605e-27 # Mean molecular mass of air (kg)
g = 9.81            # Gravity (m/s^2)

H = (kB * T) / (m * g)

print("Scale height (km):", H / 1000)

Scale height (km): 8.417243389278072


The standard value of atmospheric mass above sea level is: $X_0 \approx 1033.23 g/cm^2$

$$
\implies X(h) = X_0 e^{-h/H}
$$

In [7]:
# the atmospheric depth in g/cm^2 for a corresponding height h

def altitude_to_depth(h_km):
    X0 = 1033.23  # g/cm^2
    H = 8.4    # km
    X = X0 * np.exp(-h_km / H)
    return X 

In [ ]:
# map the CRII table files to the atmospheric depth

def depth_to_filename(X):
    val = int(X * 100)  # convert 0.01 g/cm² to 10
    return f"S_{val:06d}.RES" # then creates the file name S_000010.RES

# depth_to_filename(0.1)

'S_000010.RES'

In [27]:
# Load one CRII table

def load_crii_table(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
        # print(lines)
    # First line: Phi values
    phi_vals = np.array([float(x) for x in lines[0][2:].split()])
    
    Pc_vals = []
    CRII = []
    
    for line in lines[1:]:
        parts = line.split()
        Pc_vals.append(float(parts[0]))
        CRII.append([float(x) for x in parts[1:]])
    
    return np.array(Pc_vals), phi_vals, np.array(CRII)

Pc_vals, phi_vals, CRII = load_crii_table("data/CRII_tables/S_000001.RES")
print(f"Pc values: {Pc_vals}\n Phi values: {phi_vals} \n CRII values: {CRII}")

Pc values: [ 0.1  0.3  0.5  0.7  0.9  1.1  1.3  1.5  1.7  1.9  2.1  2.3  2.5  2.7
  2.9  3.1  3.3  3.5  3.7  3.9  4.1  4.3  4.5  4.7  4.9  5.1  5.3  5.5
  5.7  5.9  6.1  6.3  6.5  6.7  6.9  7.1  7.3  7.5  7.7  7.9  8.1  8.3
  8.5  8.7  8.9  9.1  9.3  9.5  9.7  9.9 10.1 10.3 10.5 10.7 10.9 11.1
 11.3 11.5 11.7 11.9 12.1 12.3 12.5 12.7 12.9 13.1 13.3 13.5 13.7 13.9
 14.1 14.3 14.5 14.7 14.9 16.9 20. ]
 Phi values: [   0.   50.  100.  150.  200.  250.  300.  350.  400.  450.  500.  550.
  600.  650.  700.  750.  800.  850.  900.  950. 1000. 1050. 1100. 1150.
 1200. 1250. 1300. 1350. 1400. 1450. 1500.] 
 CRII values: [[1973000. 1066000.  766200. ...   49940.   47450.   45130.]
 [1472000.  963500.  715700. ...   49750.   47270.   44980.]
 [1001000.  748600.  590300. ...   49010.   46600.   44370.]
 ...
 [   5939.    5881.    5821. ...    4550.    4509.    4469.]
 [   4806.    4764.    4722. ...    3795.    3765.    3735.]
 [   3644.    3616.    3590. ...    2979.    2959.    2939.]]


In [28]:
# interpolates the value of crii in case the phi or Pc is not in teh table
from scipy.interpolate import RegularGridInterpolator

def build_crii_interpolator(Pc_vals, phi_vals, CRII):
    return RegularGridInterpolator((Pc_vals, phi_vals), CRII)

interp = RegularGridInterpolator(
    (Pc_vals, phi_vals),  # grid axes
    CRII                  # table values
)

In [ ]:
# compute CRII at a specific location

def compute_crii(Pc, phi, interp):
    return float(interp([[Pc, phi]])[0])

compute_crii(0.2, 25, interp) # the value should be between  1973000. and 1066000. for 0.01 g/cm^2 or S_000001.RES file

1368624.9999999998

In [33]:
from OTSO import cutoff
import pandas as pd
# Use the existing dataset `data/rigidity_cutoff_values.csv` to obtain the Rc value

df = pd.read_csv("data/rigidity_cutoff_values.csv")

lats = np.sort(df["Latitude"].unique())
lons = np.sort(df["Longitude"].unique())
alts = np.sort(df["Altitude"].unique())

Rc_grid = np.zeros((len(lats), len(lons), len(alts)))

for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        for k, alt in enumerate(alts):

            val = df[
                (df["Latitude"] == lat) &
                (df["Longitude"] == lon) &
                (df["Altitude"] == alt)
            ]["Rc"].values

            Rc_grid[i, j, k] = val[0]

In [ ]:
# interpolate the latitude, longitude, and altitude values 

from scipy.interpolate import RegularGridInterpolator

Rc_interp = RegularGridInterpolator(
    (lats, lons, alts),
    Rc_grid,
    bounds_error=False,
    fill_value=None
)

In [35]:
def get_Rc(lat, lon, alt):
    return Rc_interp([[lat, lon, alt]])[0]

In [ ]:
# Obtain the CRII value at a specific place by interpolation

def get_crii(lat, lon, alt, phi, interp_crii):
    
    Rc = get_Rc(lat, lon, alt)
    
    return compute_crii(Rc, phi, interp_crii)

In [ ]:
get_crii(30, 60, 30, 512, interp) # should be between 7144 and 7064 for 0.01g/cm^2 (S_000010.RES)

7018.736